# 07 - Context Claims Verification (Macro + Peer Lens)

This notebook evaluates context claims using live DeFiLlama endpoints.

**Source class:** API_DERIVED (external market context)


## Claims-oriented checks

- What is current aggregate lending TVL?
- Where does Lazy Summer sit in category rankings?
- How does Lazy Summer compare to Summer.fi broader scope?

This notebook avoids on-chain proof claims and keeps context metrics explicitly separated.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
import requests

REFRESH_CONTEXT = True

if REFRESH_CONTEXT:
    protocols = requests.get("https://api.llama.fi/protocols", timeout=60).json()
    lazy_protocol = requests.get("https://api.llama.fi/protocol/lazy-summer-protocol", timeout=60).json()
    summer_protocol = requests.get("https://api.llama.fi/protocol/summer.fi", timeout=60).json()
else:
    protocols = []
    lazy_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "tvl")
    summer_path = nbx.latest_defillama_snapshot("summer.fi", "tvl")
    lazy_protocol = nbx.load_json(lazy_path) if lazy_path else {}
    summer_protocol = nbx.load_json(summer_path) if summer_path else {}

protocols_df = pd.DataFrame(protocols)
if not protocols_df.empty:
    protocols_df["tvl"] = pd.to_numeric(protocols_df.get("tvl"), errors="coerce").fillna(0.0)
    protocols_df["category"] = protocols_df.get("category", "").fillna("")

def latest_tvl_from_protocol_payload(payload: dict) -> float:
    tvl_df = pd.DataFrame(payload.get("tvl") or [])
    if tvl_df.empty or "totalLiquidityUSD" not in tvl_df.columns:
        return np.nan
    series = pd.to_numeric(tvl_df["totalLiquidityUSD"], errors="coerce").dropna()
    return float(series.iloc[-1]) if not series.empty else np.nan

lazy_tvl_latest = latest_tvl_from_protocol_payload(lazy_protocol)
summer_tvl_latest = latest_tvl_from_protocol_payload(summer_protocol)

print("Lazy latest TVL:", lazy_tvl_latest)
print("Summer.fi latest TVL:", summer_tvl_latest)


In [ ]:
if protocols_df.empty:
    raise RuntimeError("`protocols` dataset is empty; check network/API availability.")

lending_df = protocols_df[protocols_df["category"].str.lower() == "lending"].copy()
lending_total_tvl = float(lending_df["tvl"].sum())

yield_agg_df = protocols_df[protocols_df["category"].str.lower() == "yield aggregator"].copy()
yield_agg_df = yield_agg_df.sort_values("tvl", ascending=False).reset_index(drop=True)
lazy_rank = None
if not yield_agg_df.empty and "slug" in yield_agg_df.columns:
    match = yield_agg_df[yield_agg_df["slug"] == "lazy-summer-protocol"]
    if not match.empty:
        lazy_rank = int(match.index[0] + 1)

context_summary = pd.DataFrame([
    {
        "lending_total_tvl_usd": lending_total_tvl,
        "lazy_tvl_latest_usd": lazy_tvl_latest,
        "summer_tvl_latest_usd": summer_tvl_latest,
        "lazy_rank_in_yield_aggregators_by_tvl": lazy_rank,
    }
])

display(context_summary)

top_lending = lending_df.sort_values("tvl", ascending=False).head(15)
display(top_lending[["name", "slug", "tvl"]].head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

top_lending = lending_df.sort_values("tvl", ascending=False).head(12)
axes[0].barh(top_lending["name"], top_lending["tvl"], color="#4c78a8")
axes[0].set_title("Top Lending Protocols by TVL")
axes[0].set_xscale("log")
axes[0].invert_yaxis()

scope_df = pd.DataFrame(
    {
        "scope": ["Lazy Summer", "Summer.fi"],
        "latest_tvl_usd": [lazy_tvl_latest, summer_tvl_latest],
    }
)
axes[1].bar(scope_df["scope"], scope_df["latest_tvl_usd"], color=["#2a9d8f", "#9c6644"])
axes[1].set_title("Scope Comparison: Lazy Summer vs Summer.fi")
axes[1].set_ylabel("TVL (USD)")

plt.show()


In [ ]:
claims_table = pd.DataFrame([
    {
        "claim": "Lending TVL is around a fixed value",
        "current_observation": lending_total_tvl,
        "interpretation": "Time-varying; must always quote date + source.",
    },
    {
        "claim": "Lazy Summer and Summer.fi TVL are interchangeable",
        "current_observation": f"Lazy={lazy_tvl_latest:,.0f}, Summer.fi={summer_tvl_latest:,.0f}",
        "interpretation": "False. Distinct scopes; do not conflate.",
    },
])

display(claims_table)
